In [1]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(f'Tensorflow version: {tf.version.VERSION}')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device: ', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))


Num GPUs Available:  0
Tensorflow version: 2.21.0
Using device:  cuda
NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
try:
    import kagglehub
except ImportError:
    !pip install kagglehub

from src.dataset.DroneSegDataSet import MyDataset
from src.dataset.CheckDataset import check_dataset


当前脚本路径: F:\2025ProjectDroneSeg\2026.3.25_BBC6521_DroneSegModel\src\dataset
项目根路径: F:\2025ProjectDroneSeg\2026.3.25_BBC6521_DroneSegModel


In [3]:
def print_model_params(model):
    print("Model parameter summary:")
    print("-" * 60)
    total = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            numel = param.numel()
            total += numel
            print(f"{name:<50} {numel:>12,}")
    print("-" * 60)
    print(f"{'Total trainable parameters':<50} {total:>12,}")


In [4]:
# 检查硬件环境
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from src.model.Model2.DroneSegModel import DroneSegModel

model = DroneSegModel().to(device)


Using device: cuda


In [5]:
from torchvision import transforms
# 加载数据集
ds_path = check_dataset()
dataset = MyDataset(
    'drone_seg_dataset/classes_dataset/classes_dataset/original_images',
    'drone_seg_dataset/classes_dataset/classes_dataset/label_images_semantic',
    transform=transforms.Compose([
        transforms.ToTensor(),
    ])
)

from torch.utils.data import random_split, DataLoader
import numpy as np

train_set, test_set = random_split(dataset, [int(0.9 * len(dataset)), len(dataset) - int(0.9 * len(dataset))])
train_loader = DataLoader(dataset=train_set, batch_size=2, shuffle=True)

print_model_params(model)


Dataset already exists.
成功匹配文件: 375.png, 543.png, 461.png 等 400 个文件
数据集总样本数: 400
训练集样本数: 360
测试集样本数: 40
Train set - img shape: torch.Size([22, 736, 960]), lbl shape: torch.Size([736, 960])
Train set - unique labels in lbl0: tensor([0, 2, 3, 4])


In [1]:
from src.training.train_Model2.TrainPhase import train_phase

def train_all_phases(
        model,
        dataloader,
        criterion,
        train_config,
        logger,
        device,
):

    for phase_idx, (config) in enumerate(train_config):

        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=config['lr'],
            momentum=config['momentum'],
        )

        print(f"Phase {phase_idx} Starting | lr: {config['lr']}, momentum: {config['momentum']}, epochs: {config['epochs']}")
        logger.start_new_phase(f"Phase {phase_idx}")

        train_phase(
            model=model,
            dataloader=dataloader,
            logger=logger,
            epochs=config['epochs'],
            sample_period=min(50, config['epochs'] // 2),  # 每个阶段至少保存 2 次样本
            device=device,
            logging_info={
                'phase': phase_idx,
            },
            criterion=criterion,
            optimizer=optimizer,
        )

        logger.end_current_phase(model_to_save=model)

    logger.finalize_and_plot_all()
    torch.cuda.empty_cache()

In [ ]:
INIT_MODEL_PATH = "" # 在此处填入.pth文件路径以加载预训练权重，留空则从头开始训练
if INIT_MODEL_PATH:
    model.load_state_dict(torch.load(INIT_MODEL_PATH, map_location=device))
    print(f"已从 '{INIT_MODEL_PATH}' 加载模型参数。")
else:
    print("未指定预训练模型，从头开始训练。")

from src.logging.Logger import Logger
print("Train logger: Unet_Segmentation_Logger")
logger = Logger(
    title='Model2',
    log_format=[
        'phase', 'epoch', 'elapsed_time', 'loss', 'miou', 'accuracy', 'time'
    ],
    output_frequency=1,
)

# --- 必须定义类别数量 ---
NUM_CLASSES = 5 # 请将此值替换为您数据集的实际类别数量，例如PASCAL VOC有21类

# --- 定义损失函数和优化器 ---
criterion = torch.nn.CrossEntropyLoss()
# 优化器使用 SGD，学习率与动量动态调整
train_config = [
    {'lr': 0.2, 'momentum': 0.85, 'epochs': 40},
    {'lr': 0.01, 'momentum': 0.9, 'epochs': 50},

    {'lr': 0.05, 'momentum': 0.9, 'epochs': 40},
    {'lr': 0.005, 'momentum': 0.9, 'epochs': 30},

    {'lr': 0.01, 'momentum': 0.9, 'epochs': 40},
    {'lr': 0.0005, 'momentum': 0.95, 'epochs': 25},
]

train_all_phases(
    model=model,
    dataloader=train_loader,
    criterion=criterion,
    train_config=train_config,
    logger=logger,
    device=device,
)